# Data Warehouse Queries - Retail Price Analysis
## Analytical Queries for Naivas & Quickmart Price Data

This notebook contains SQL queries for analyzing the retail price data warehouse.

**Prerequisites**: Run `etl_pipeline.ipynb` first to load data into the warehouse.

---

## Setup - Connect to Database

In [14]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

# Connect to data warehouse
conn = sqlite3.connect('../data_warehouse.db')
print("✓ Connected to data_warehouse.db")

# Verify connection
query = "SELECT name FROM sqlite_master WHERE type='table'"
tables = pd.read_sql_query(query, conn)
print("\nAvailable Tables:")
print(tables)

✓ Connected to data_warehouse.db

Available Tables:
              name
0       dim_stores
1  sqlite_sequence
2   dim_categories
3     dim_products
4        dim_dates
5      fact_prices


## Query 1: Data Warehouse Summary

In [15]:
query = """
SELECT 
    'Total Price Observations' as metric,
    COUNT(*) as value
FROM fact_prices
UNION ALL
SELECT 
    'Unique Products',
    COUNT(*)
FROM dim_products
UNION ALL
SELECT 
    'Stores',
    COUNT(*)
FROM dim_stores
UNION ALL
SELECT 
    'Categories',
    COUNT(*)
FROM dim_categories
UNION ALL
SELECT 
    'Date Range (Days)',
    COUNT(*)
FROM dim_dates
"""

df_summary = pd.read_sql_query(query, conn)
print("="*50)
print("Data Warehouse Summary")
print("="*50)
print(df_summary.to_string(index=False))

Data Warehouse Summary
                  metric  value
Total Price Observations 622468
         Unique Products  20232
                  Stores      2
              Categories      8
       Date Range (Days)     76


## Query 2: Price Details View

In [16]:
query = """
SELECT * FROM vw_price_details
LIMIT 20
"""

df_price_details = pd.read_sql_query(query, conn)
print("Sample Price Details:")
print(df_price_details)

Sample Price Details:
    price_id        date  year  month  quarter store_name  \
0          1  2026-01-01  2026      1        1     Naivas   
1          2  2026-01-01  2026      1        1     Naivas   
2          3  2026-01-01  2026      1        1     Naivas   
3          4  2026-01-01  2026      1        1     Naivas   
4          5  2026-01-01  2026      1        1     Naivas   
5          6  2026-01-01  2026      1        1     Naivas   
6          7  2026-01-01  2026      1        1     Naivas   
7          8  2026-01-01  2026      1        1     Naivas   
8          9  2026-01-01  2026      1        1     Naivas   
9         10  2026-01-01  2026      1        1     Naivas   
10        11  2026-01-01  2026      1        1     Naivas   
11        12  2026-01-01  2026      1        1     Naivas   
12        13  2026-01-01  2026      1        1     Naivas   
13        14  2026-01-01  2026      1        1     Naivas   
14        15  2026-01-01  2026      1        1     Naivas   
15

## Query 3: Daily Average Prices by Store

In [17]:
query = """
SELECT 
    date,
    store_name,
    product_count,
    ROUND(avg_price, 2) as avg_price,
    ROUND(min_price, 2) as min_price,
    ROUND(max_price, 2) as max_price
FROM vw_daily_avg_prices
ORDER BY date DESC, store_name
LIMIT 30
"""

df_daily_avg = pd.read_sql_query(query, conn)
print("Daily Average Prices by Store:")
print(df_daily_avg)

Daily Average Prices by Store:
          date store_name  product_count  avg_price  min_price  max_price
0   2026-04-23  Quickmart           5142    1098.88        5.0   113000.0
1   2026-04-22  Quickmart           3141    1650.41        5.0   113000.0
2   2026-04-21  Quickmart           5165    1087.52        4.0   113000.0
3   2026-04-20  Quickmart           9380    1297.10        1.0   113000.0
4   2026-04-16  Quickmart          10162    1663.32        1.0   142995.0
5   2026-04-15  Quickmart          10774    1573.03        1.0   142995.0
6   2026-04-14  Quickmart          10524    1598.57        1.0   142995.0
7   2026-04-13     Naivas            893    6915.72       15.0   216995.0
8   2026-04-13  Quickmart          10591    1608.42        1.0   142995.0
9   2026-04-10  Quickmart          10584    1614.66        1.0   199995.0
10  2026-04-09  Quickmart           8374    1066.49        1.0   113000.0
11  2026-04-08  Quickmart          10421    1591.15        1.0   199995.0
12  202

## Query 4: Product Price Comparison Across Stores

In [18]:
query = """
SELECT 
    product_name,
    category_name,
    store_name,
    ROUND(avg_price, 2) as avg_price,
    observation_count
FROM vw_product_price_comparison
WHERE observation_count > 5
ORDER BY product_name, store_name
LIMIT 50
"""

df_comparison = pd.read_sql_query(query, conn)
print("Product Price Comparison:")
print(df_comparison)

Product Price Comparison:
                             product_name    category_name store_name  \
0                                  #Name?  Household Items  Quickmart   
1      -Paris Scrabble Game B/S Ns 230997  Household Items  Quickmart   
2        -Sel Stargift Round Tray 30 1223  Household Items  Quickmart   
3                           104 Shoe Rack  Household Items  Quickmart   
4          11S Doll Set (Hard Body)  053D  Household Items  Quickmart   
5                  13A Double  Socket+Usb      Electronics  Quickmart   
6   13A T/Plug Wth Fuse And Indcator(Blck      Electronics  Quickmart   
7      151 Duzzit 50Pk Multisurface Wipes    Personal Care  Quickmart   
8          151 Duzzit Metal Polish Liquid        Home Care  Quickmart   
9        151 Int.scented Dehmdifier Asst.        Home Care  Quickmart   
10              151 Interior Dehumidifier        Home Care  Quickmart   
11                       1659 Classic Red              NaN     Naivas   
12                1659 Na

## Query 5: Top 20 Most Expensive Products

In [19]:
query = """
SELECT 
    dp.product_name,
    ds.store_name,
    dc.category_name,
    ROUND(AVG(fp.price), 2) as avg_price,
    COUNT(*) as times_observed
FROM fact_prices fp
JOIN dim_products dp ON fp.product_id = dp.product_id
JOIN dim_stores ds ON fp.store_id = ds.store_id
LEFT JOIN dim_categories dc ON dp.category_id = dc.category_id
GROUP BY dp.product_name, ds.store_name, dc.category_name
ORDER BY avg_price DESC
LIMIT 20
"""

df_top_expensive = pd.read_sql_query(query, conn)
print("Top 20 Most Expensive Products:")
print(df_top_expensive)

Top 20 Most Expensive Products:
                             product_name store_name category_name  avg_price  \
0         Samsung 98" 4K Q Led Tv,S 8,S/D  Quickmart   Electronics  858920.58   
1   Hisense Tv Qled 4K Alexa Built In-100  Quickmart   Electronics  649990.00   
2                  Lg Ref Sbs Gc-X257Cqes  Quickmart   Electronics  424995.00   
3       Lg F3L2Crv2T 20/ Washer/Dryer Vcm     Naivas           NaN  285995.00   
4                       Samsung W/M Wd21T  Quickmart   Electronics  247233.71   
5                Samsung 2D S/Side Fridge     Naivas           NaN  229990.00   
6   Samsung 2D S/Side Fridge -Rs62R5005M9     Naivas           NaN  229990.00   
7           LG Oled55A26La 55 Inch4K Oled     Naivas           NaN  216995.00   
8   Lg Washer & Dryer 13/ F4Y9Ldp2Z Black     Naivas           NaN  204911.67   
9            Lg W/Dryer Fl F4V9Bdp2Ee 12/  Quickmart   Electronics  202995.00   
10        Mika Fridge Nofrost Mrnf4D490Ss  Quickmart   Electronics  199995.00

## Query 6: Top 20 Cheapest Products

In [20]:
query = """
SELECT 
    product_name,
    category_name,
    store_name,
    ROUND(avg_price, 2) as avg_price,
    observation_count
FROM vw_product_price_comparison
WHERE avg_price > 0
ORDER BY avg_price ASC
LIMIT 20
"""

df_cheapest = pd.read_sql_query(query, conn)
print("Top 20 Cheapest Products:")
print(df_cheapest)

Top 20 Cheapest Products:
                             product_name    category_name store_name  \
0                      Dimax Gift Voucher  Household Items  Quickmart   
1                      Rdl Tropical Mints            Foods  Quickmart   
2                          Big G Original            Foods  Quickmart   
3         Envelope Dl Banker Manilla(Win)  Household Items  Quickmart   
4              Envelope Dl Pocket Manilla  Household Items  Quickmart   
5       Zesta Tomato Sauce Sachets - 300X            Foods  Quickmart   
6       Economic Brown Manila Envelope C5  Household Items  Quickmart   
7           Envelope Dl Banker White Cart  Household Items  Quickmart   
8      Kenafric Ting Cola  Bubble Gum 1Pc            Foods  Quickmart   
9                   Nuvita Wafer Assorted            Foods  Quickmart   
10                       Teeka Gums Fruit            Foods  Quickmart   
11                        Teeka Gums Mint            Foods  Quickmart   
12  Kenafric Yoghurtassor

## Query 7: Monthly Price Trends

In [21]:
query = """
SELECT 
    dd.year,
    dd.month,
    dd.month_name,
    ds.store_name,
    COUNT(*) as product_count,
    ROUND(AVG(fp.price), 2) as avg_price,
    ROUND(MIN(fp.price), 2) as min_price,
    ROUND(MAX(fp.price), 2) as max_price
FROM fact_prices fp
JOIN dim_dates dd ON fp.date_id = dd.date_id
JOIN dim_stores ds ON fp.store_id = ds.store_id
GROUP BY dd.year, dd.month, dd.month_name, ds.store_name
ORDER BY dd.year, dd.month, ds.store_name
"""

df_monthly = pd.read_sql_query(query, conn)
print("Monthly Price Trends:")
print(df_monthly)

Monthly Price Trends:
   year  month month_name store_name  product_count  avg_price  min_price  \
0  2026      1    January     Naivas          81700    2333.87        6.0   
1  2026      1    January  Quickmart          66538    2060.94        4.0   
2  2026      2   February     Naivas          58691    2750.36        6.0   
3  2026      2   February  Quickmart         108504    3206.46        4.0   
4  2026      3      March     Naivas          52397    3047.18        8.0   
5  2026      3      March  Quickmart         159487    2877.67        4.0   
6  2026      4      April     Naivas            893    6915.72       15.0   
7  2026      4      April  Quickmart          94258    1473.92        1.0   

   max_price  
0   285995.0  
1   649990.0  
2   285995.0  
3  1099995.0  
4   229990.0  
5  1099995.0  
6   216995.0  
7   199995.0  


## Query 8: Category Analysis

In [22]:
query = """
SELECT 
    category_name,
    COUNT(DISTINCT product_name) as unique_products,
    COUNT(*) as total_observations,
    ROUND(AVG(price), 2) as avg_price,
    ROUND(MIN(price), 2) as min_price,
    ROUND(MAX(price), 2) as max_price
FROM vw_price_details
WHERE category_name IS NOT NULL
GROUP BY category_name
ORDER BY avg_price DESC
"""

df_categories = pd.read_sql_query(query, conn)
print("Category Analysis:")
print(df_categories)

Category Analysis:
     category_name  unique_products  total_observations  avg_price  min_price  \
0      Electronics             1096               32190   25983.57       47.0   
1     Landing_Page              157                4546    3263.91       39.0   
2          Textile              303                5177    2283.39       15.0   
3  Household Items             3052               66061    1393.71        1.0   
4    Personal Care             2793               81350     582.99       13.0   
5        Home Care             1456               49192     448.77        7.0   
6            Foods             5128              199522     323.61        4.0   
7    Fresh Produce              459               13534     317.59       20.0   

   max_price  
0  1099995.0  
1   132700.0  
2    41995.0  
3    99999.0  
4     6654.0  
5    16495.0  
6    11489.0  
7     2799.0  


## Query 9: Store Price Comparison (Side-by-Side)

In [23]:
query = """
SELECT 
    product_name,
    category_name,
    MAX(CASE WHEN store_name = 'Naivas' THEN avg_price END) as naivas_price,
    MAX(CASE WHEN store_name = 'Quickmart' THEN avg_price END) as quickmart_price,
    ROUND(
        MAX(CASE WHEN store_name = 'Naivas' THEN avg_price END) - 
        MAX(CASE WHEN store_name = 'Quickmart' THEN avg_price END), 
        2
    ) as price_difference,
    CASE 
        WHEN MAX(CASE WHEN store_name = 'Naivas' THEN avg_price END) < 
             MAX(CASE WHEN store_name = 'Quickmart' THEN avg_price END) 
        THEN 'Naivas Cheaper'
        ELSE 'Quickmart Cheaper'
    END as cheaper_store
FROM vw_product_price_comparison
WHERE observation_count > 3
GROUP BY product_name, category_name
HAVING naivas_price IS NOT NULL AND quickmart_price IS NOT NULL
ORDER BY ABS(price_difference) DESC
LIMIT 30
"""

df_store_comparison = pd.read_sql_query(query, conn)
print("Store Price Comparison:")
print(df_store_comparison)

Store Price Comparison:
                          product_name  category_name  naivas_price  \
0                      Sunrice Basmati          Foods   8050.000000   
1   Mika Microwave Mmwdgpb2075Mb Black    Electronics  12841.875000   
2                Fresh Fri Cooking Oil          Foods    190.000000   
3                   Pearl Pishori Rice          Foods   1339.888889   
4        Pietro Extra Virgin Olive Oil          Foods   2206.598639   
5           Haagen Dazs Salted Caramel          Foods   1450.000000   
6                   Rina Vegetable Oil          Foods    741.694030   
7           Lyons Strawberry Ice Cream          Foods    620.000000   
8              Lyons Vanilla Ice Cream          Foods    870.000000   
9   Mika Juicer 4In1 500W 2Spd Mjr411W    Electronics   7540.918367   
10      Oilyssa Extra Virgin Olive Oil          Foods   1894.642857   
11       Naturalli Baobab Fruit Powder          Foods    899.000000   
12                  Jamii Pishori Rice          Foods

## Query 10: Price Volatility Analysis

In [24]:
query = """
SELECT 
    product_name,
    category_name,
    store_name,
    COUNT(*) as observations,
    ROUND(MIN(price), 2) as min_price,
    ROUND(MAX(price), 2) as max_price,
    ROUND(MAX(price) - MIN(price), 2) as price_range,
    ROUND(AVG(price), 2) as avg_price,
    ROUND(((MAX(price) - MIN(price)) / AVG(price)) * 100, 2) as volatility_pct
FROM vw_price_details
GROUP BY product_name, category_name, store_name
HAVING observations > 5
ORDER BY volatility_pct DESC
LIMIT 30
"""

df_volatility = pd.read_sql_query(query, conn)
print("Products with Highest Price Volatility:")
print(df_volatility)

Products with Highest Price Volatility:
                               product_name    category_name store_name  \
0                      Toogoody Toffee Bars            Foods  Quickmart   
1                     Exe All Purpose Flour     Landing_Page  Quickmart   
2                    Manji Milkstar Biscuit              NaN     Naivas   
3                              Mumias Sugar            Foods  Quickmart   
4                        Fanta Blackcurrant            Foods  Quickmart   
5                     Royco Mchuzi Mix Beef            Foods  Quickmart   
6                              Fanta Orange            Foods  Quickmart   
7                  Eris Surgical Spirit 70%        Home Care  Quickmart   
8             Rollana Chocolate Wafer Balls            Foods  Quickmart   
9               Rosapharm Methylated Spirit        Home Care  Quickmart   
10                        Pembe Maize Flour            Foods  Quickmart   
11                            Fanta Passion            Foods

## Query 11: Weekend vs Weekday Pricing

In [25]:
query = """
SELECT 
    store_name,
    CASE WHEN is_weekend = 1 THEN 'Weekend' ELSE 'Weekday' END as day_type,
    COUNT(*) as observations,
    ROUND(AVG(price), 2) as avg_price,
    ROUND(MIN(price), 2) as min_price,
    ROUND(MAX(price), 2) as max_price
FROM vw_price_details
GROUP BY store_name, day_type
ORDER BY store_name, day_type
"""

df_weekend = pd.read_sql_query(query, conn)
print("Weekend vs Weekday Pricing:")
print(df_weekend)

DatabaseError: Execution failed on sql '
SELECT 
    store_name,
    CASE WHEN is_weekend = 1 THEN 'Weekend' ELSE 'Weekday' END as day_type,
    COUNT(*) as observations,
    ROUND(AVG(price), 2) as avg_price,
    ROUND(MIN(price), 2) as min_price,
    ROUND(MAX(price), 2) as max_price
FROM vw_price_details
GROUP BY store_name, day_type
ORDER BY store_name, day_type
': no such column: is_weekend

## Query 12: Most Frequently Observed Products

In [26]:
query = """
SELECT 
    product_name,
    category_name,
    COUNT(*) as total_observations,
    COUNT(DISTINCT store_name) as stores_available,
    ROUND(AVG(price), 2) as avg_price,
    ROUND(MIN(price), 2) as min_price,
    ROUND(MAX(price), 2) as max_price
FROM vw_price_details
GROUP BY product_name, category_name
ORDER BY total_observations DESC
LIMIT 30
"""

df_most_observed = pd.read_sql_query(query, conn)
print("Most Frequently Observed Products:")
print(df_most_observed)

Most Frequently Observed Products:
                        product_name  category_name  total_observations  \
0                  Daima Yoghurt Cup          Foods                 722   
1                 Rina Vegetable Oil          Foods                 557   
2                      Flamingo Soap  Personal Care                 539   
3                            Ramtons    Electronics                 505   
4                             Dettol  Personal Care                 493   
5               Rinsun Sunflower Oil          Foods                 492   
6                              Armco    Electronics                 489   
7                   Elianto Corn Oil          Foods                 436   
8                     Gnomis Far Far          Foods                 387   
9              Santa Lucia Spaghetti          Foods                 380   
10              Kentaste Coconut Oil          Foods                 359   
11                    Gnomis Popcorn          Foods              

## Custom Query Section
Write your own SQL queries here:

In [27]:
# Example: Search for specific product
product_search = "Maize"  # Change this to search for different products

query = f"""
SELECT 
    date,
    product_name,
    store_name,
    price,
    category_name
FROM vw_price_details
WHERE product_name LIKE '%{product_search}%'
ORDER BY date DESC, store_name
LIMIT 50
"""

df_custom = pd.read_sql_query(query, conn)
print(f"Search Results for '{product_search}':")
print(df_custom)

Search Results for 'Maize':
          date                              product_name store_name  price  \
0   2026-04-23  Ajab Premium Fortified Sifted Maize Meal  Quickmart  162.0   
1   2026-04-23                          Soko Maize Flour  Quickmart  161.0   
2   2026-04-23                              Amaize Flour  Quickmart  197.0   
3   2026-04-23                          Jogoo Maize Meal  Quickmart  168.0   
4   2026-04-23                           Dola Maize Meal  Quickmart  164.0   
5   2026-04-23                         Jogoo Maize Flour  Quickmart  860.0   
6   2026-04-23                 Spenza Premium Maize Meal  Quickmart  187.0   
7   2026-04-23                         Joymax Maize Meal  Quickmart  140.0   
8   2026-04-23                          Soko Maize Flour  Quickmart  847.0   
9   2026-04-23                   Raha Premium Maize Meal  Quickmart  197.0   
10  2026-04-23                         Pembe Maize Flour  Quickmart  415.0   
11  2026-04-23                      

## Cleanup - Close Database Connection

In [1]:
# Close connection
conn.close()
print("✓ Database connection closed")

NameError: name 'conn' is not defined

---
## Next Steps

- **Visualizations**: Create charts using matplotlib/seaborn
- **Advanced Analytics**: Time series forecasting, price prediction
- **Export Reports**: Save query results to Excel/CSV
- **More Queries**: Check `scripts/example_queries.py` for additional examples